In [3]:
from pathlib import Path
from awpy import Demo

demos = sorted(Path("../../data/demos").glob("*.dem"))
demos

[WindowsPath('../../data/demos/1-12faf4de-9145-4498-ae14-ddcc857a4e53-1-1.dem'),
 WindowsPath('../../data/demos/1-51a7354f-eb4a-4e81-b7a1-8f146899f55b-1-1.dem'),
 WindowsPath('../../data/demos/1-845fdd48-e3bd-4485-aacd-d0d0d6a4ed99-1-1.dem'),
 WindowsPath('../../data/demos/1-ded52a9e-1ed9-4c75-9e82-8376d3e8c9ae-1-1.dem'),
 WindowsPath('../../data/demos/dd900960fd49.dem')]

In [ ]:
dem = Demo(demos[-1], verbose=True)
dem.parse()
dem.header
print(dem.kills.columns)
print(dem.damages.columns)
print(dem.damages["weapon"].unique().to_list())
print(dem.events.keys())

In [ ]:
for name in ["rounds", "kills", "damages", "shots", "grenades", "ticks"]:
    df = getattr(dem, name)
    print(f"{name}: {df.shape}")
    print(df.columns)
    print()

In [ ]:
dem.rounds.head(5)

In [ ]:
dem.kills.head(5)

In [ ]:
dem.kills.group_by("attacker_name").len().sort("len", descending=True)

In [ ]:
import polars as pl

n_rounds = dem.rounds.height
kills = dem.kills
dmg = dem.damages.filter(pl.col("attacker_side") != pl.col("victim_side"))  # sans friendly fire

k = kills.group_by("attacker_name").agg(
    pl.len().alias("kills"),
    pl.col("headshot").sum().alias("hs"),
)
d = kills.group_by("victim_name").agg(pl.len().alias("deaths"))
adr = dmg.group_by("attacker_name").agg(
    (pl.col("dmg_health_real").sum() / n_rounds).alias("adr")
)

stats = (
    k.rename({"attacker_name": "name"})
    .join(d.rename({"victim_name": "name"}), on="name", how="full", coalesce=True)
    .join(adr.rename({"attacker_name": "name"}), on="name", how="full", coalesce=True)
    .fill_null(0)
    .with_columns((pl.col("hs") / pl.col("kills")).alias("hs_pct"))
    .sort("kills", descending=True)
)
stats

In [ ]:
print(getattr(dem, "tickrate", "attribut absent"))

no_plant = dem.rounds.filter(pl.col("bomb_plant").is_null())
(
    no_plant.with_columns(
        ((pl.col("end") - pl.col("freeze_end")) / 115).alias("ticks_per_s")
    )
    .select("round_num", "reason", "ticks_per_s")
    .sort("ticks_per_s", descending=True)
)

In [ ]:
TICKRATE = dem.tickrate
TRADE_WINDOW_S = 3.0 # secondes
TRADE_WINDOW = int(TRADE_WINDOW_S * TICKRATE)

# Morts causées par un adversaire (on exclut le friendly fire et les morts "monde")
deaths = (
    dem.kills
    .filter(
        pl.col("attacker_steamid").is_not_null()
        & (pl.col("attacker_side") != pl.col("victim_side"))
    )
    .select(
        "round_num",
        pl.col("tick").alias("death_tick"),
        "victim_steamid", "victim_name", "victim_side",
        pl.col("attacker_steamid").alias("killer_steamid"),
    )
    .with_row_index("death_id")
)

# Toutes les kills, vues comme des "vengeances" potentielles
revenges = dem.kills.select(
    "round_num",
    pl.col("tick").alias("revenge_tick"),
    pl.col("victim_steamid").alias("killer_steamid"),   # le tueur initial qui meurt
    pl.col("attacker_side").alias("revenger_side"),     # camp de celui qui le tue
)

# Mort tradée = le tueur est abattu par un coéquipier de la victime dans la fenêtre
traded_ids = (
    deaths.join(revenges, on=["round_num", "killer_steamid"])
    .filter(
        (pl.col("revenger_side") == pl.col("victim_side"))
        & (pl.col("revenge_tick") >= pl.col("death_tick"))
        & (pl.col("revenge_tick") - pl.col("death_tick") <= TRADE_WINDOW)
    )
    .select("death_id")
    .unique()
)

deaths = deaths.with_columns(
    pl.col("death_id").is_in(traded_ids["death_id"]).alias("traded")
)

trade_stats = (
    deaths.group_by("victim_steamid", "victim_name")
    .agg(
        pl.len().alias("deaths"),
        pl.col("traded").sum().alias("traded"),
    )
    .with_columns((pl.col("traded") / pl.col("deaths")).alias("traded_pct"))
    .sort("deaths", descending=True)
)
trade_stats

In [ ]:
delays = (
    deaths.join(revenges, on=["round_num", "killer_steamid"])
    .filter(
        (pl.col("revenger_side") == pl.col("victim_side"))
        & (pl.col("revenge_tick") >= pl.col("death_tick"))
    )
    .with_columns(
        ((pl.col("revenge_tick") - pl.col("death_tick")) / TICKRATE).alias("delay_s")
    )
    .group_by("death_id")
    .agg(pl.col("delay_s").min())
)

deaths_delay = deaths.join(delays, on="death_id", how="left")

import sys
sys.path.append("..")
from app.analysis.trades import compute_trades

trades = compute_trades(dem.kills, tickrate=dem.tickrate, window_s=3.0)

# Vérifier que le module donne les mêmes résultats que le notebook
with pl.Config(tbl_rows=-1, tbl_cols=-1):
    display(
        deaths_delay.filter(pl.col("victim_name") == "WilloKage")
        .select("round_num", "death_tick", "delay_s", "traded")
    )
    display(
        trades.filter(pl.col("victim_name") == "WilloKage")
        .select("round_num", "death_tick", "delay_s", "traded")
    )

In [ ]:
with pl.Config(tbl_rows=-1, tbl_cols=-1):
    display(deaths.filter(pl.col("victim_name") == "WilloKage"))

In [ ]:
from app.analysis.stats import compute_player_stats

players = dem.ticks.group_by("steamid").agg(pl.col("name").last())
stats = compute_player_stats(dem.kills, dem.damages, dem.rounds, players)

unattributed = dem.kills["victim_steamid"].null_count()
with pl.Config(tbl_rows=-1, tbl_cols=-1):
    display(stats)
print("morts sans victime identifiée :", unattributed)
print("contrôle :", stats["deaths"].sum() + unattributed == dem.kills.height)

In [ ]:
ME = "WilloKage"
mine = dem.kills.filter(pl.col("attacker_name") == ME)

print("kills bruts dans la table kills :", mine.height)
print("dont camp attaquant nul :", mine.filter(pl.col("attacker_side").is_null()).height)
print("dont camp victime nul   :", mine.filter(pl.col("victim_side").is_null()).height)
print("dont même camp          :", mine.filter(pl.col("attacker_side") == pl.col("victim_side")).height)

with pl.Config(tbl_rows=-1):
    display(
        dem.kills.group_by("attacker_steamid", "attacker_name")
        .len()
        .sort("len", descending=True)
    )

In [5]:
import polars as pl
pl.read_parquet(Path("../../data/matches/dd900960fd49/players.parquet"))
pl.read_parquet(Path("../../data/matches/dd900960fd49/kills.parquet")).select("attacker_name", "victim_name", "attacker_steamid").head()

attacker_name,victim_name,attacker_steamid
str,str,u64
"""Estiroides""","""anita2232""",76561198002719831
"""WilloKage""","""PAXIKO""",76561198996627086
"""WilloKage""","""bkfrm8""",76561198996627086
"""K0M0RNIK_""","""EXO4""",76561199028195165
"""WilloKage""","""M4rc1nparuwa""",76561198996627086


In [6]:
pl.read_parquet(Path("../demo_data/de0de0de0de0/players.parquet"))
pl.read_parquet(Path("../demo_data/de0de0de0de0/kills.parquet")).select("attacker_name", "victim_name", "attacker_steamid").head()

attacker_name,victim_name,attacker_steamid
str,str,u64
"""Teammate 4""","""Opponent 3""",1010
"""You""","""Opponent 4""",1003
"""You""","""Opponent 1""",1003
"""Opponent 2""","""Teammate 1""",1002
"""You""","""Opponent 5""",1003


In [7]:
import polars as pl
from pathlib import Path

src = Path(r"D:\perso\projets\cs-demo-dashboard\data\matches\dd900960fd49")
dst = Path("../demo_data/de0de0de0de0")

orig = pl.read_parquet(src / "players.parquet")
ids, names = orig["steamid"].to_list(), orig["name"].to_list()

leaks = []
for f in dst.glob("*.parquet"):
    df = pl.read_parquet(f)
    for col, dt in df.schema.items():
        if dt == pl.String and df[col].is_in(names).any():
            leaks.append((f.name, col))
        elif dt.is_integer() and df[col].is_in(ids).any():
            leaks.append((f.name, col))
print(leaks or "aucune fuite")
print((dst / "meta.json").read_text())

aucune fuite
{
  "id": "de0de0de0de0",
  "owner": "1003",
  "filename": "demo.dem",
  "status": "ready",
  "map_name": "de_cache",
  "tickrate": 128,
  "rounds": 19,
  "error": null
}
